In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)
test_loader = DataLoader(datasets.MNIST('./data', train=False, transform=transform), batch_size=1000)

100%|██████████| 9.91M/9.91M [00:00<00:00, 35.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 954kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.19MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.71MB/s]


In [2]:
class TeacherModel(nn.Module):
    def __init__(self):
        super(TeacherModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

class StudentModel(nn.Module):
    def __init__(self):
        super(StudentModel, self).__init__()
        self.fc1 = nn.Linear(28*28, 20) 
        self.fc2 = nn.Linear(20, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [3]:
def train_knowledge_distillation(teacher, student, train_loader, optimizer, epoch, temp, alpha):
    teacher.eval() 
    student.train()
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        
        with torch.no_grad():
            teacher_preds = teacher(data)
        student_preds = student(data)
        
        student_loss = F.cross_entropy(student_preds, target)
        
        distillation_loss = nn.KLDivLoss(reduction='batchmean')(
            F.log_softmax(student_preds / temp, dim=1),
            F.softmax(teacher_preds / temp, dim=1)
        ) * (temp * temp)
        
        loss = alpha * student_loss + (1 - alpha) * distillation_loss
        
        loss.backward()
        optimizer.step()

        if batch_idx % 200 == 0:
            print(f"Train Epoch: {epoch} Loss: {loss.item():.4f}")

In [4]:
teacher = TeacherModel().to(device)
teacher_optimizer = optim.Adam(teacher.parameters(), lr=0.001)

print("--- Training Teacher Model ---")
for epoch in range(1, 3): 
    pass

student = StudentModel().to(device)
optimizer = optim.Adam(student.parameters(), lr=0.001)

temperature = 7.0
alpha = 0.1 

print("\n--- Training Student Model with Distillation ---")
for epoch in range(1, 6):
    train_knowledge_distillation(teacher, student, train_loader, optimizer, epoch, temperature, alpha)

def test(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    print(f'Accuracy: {100. * correct / len(test_loader.dataset)}%')

print("\nFinal Student Accuracy:")
test(student)

--- Training Teacher Model ---

--- Training Student Model with Distillation ---
Train Epoch: 1 Loss: 0.2746
Train Epoch: 1 Loss: 0.2051
Train Epoch: 1 Loss: 0.1990
Train Epoch: 1 Loss: 0.2017
Train Epoch: 1 Loss: 0.1979
Train Epoch: 2 Loss: 0.1984
Train Epoch: 2 Loss: 0.1951
Train Epoch: 2 Loss: 0.1984
Train Epoch: 2 Loss: 0.1985
Train Epoch: 2 Loss: 0.1982
Train Epoch: 3 Loss: 0.1997
Train Epoch: 3 Loss: 0.1995
Train Epoch: 3 Loss: 0.1969
Train Epoch: 3 Loss: 0.2014
Train Epoch: 3 Loss: 0.1987
Train Epoch: 4 Loss: 0.1958
Train Epoch: 4 Loss: 0.1989
Train Epoch: 4 Loss: 0.1979
Train Epoch: 4 Loss: 0.1951
Train Epoch: 4 Loss: 0.1992
Train Epoch: 5 Loss: 0.1983
Train Epoch: 5 Loss: 0.1966
Train Epoch: 5 Loss: 0.1971
Train Epoch: 5 Loss: 0.1990
Train Epoch: 5 Loss: 0.1987

Final Student Accuracy:
Accuracy: 90.47%
